# unimpeded Tutorial

**Universal model comparison & parameter estimation distributed over every dataset**

This notebook demonstrates how to use `unimpeded` for:
- Exploring available cosmological models and datasets
- Downloading MCMC/nested sampling chains from Zenodo
- Computing tension statistics between datasets
- Understanding different approaches to tension calculation

## Installation

For setup instructions, please refer to the [installation guide](https://unimpeded.readthedocs.io/en/latest/installation.html).

If you are reading this notebook, you have likely already completed the setup. Run the cell below to verify your installation:

In [ ]:
# Verify installation
try:
    import unimpeded
    print(f"✓ unimpeded version {unimpeded.__version__} is installed")
    print(f"✓ Installation location: {unimpeded.__file__}")
except ImportError:
    print("✗ unimpeded is not installed")
    print("\nTo install, run in your terminal:")
    print("  pip install unimpeded")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from unimpeded.database import DatabaseExplorer
from unimpeded.tension import tension_calculator, tension_stats, download_tension_inputs

# Set plotting style
%matplotlib inline

## 1. Exploring Available Data

First, let's see what models and datasets are available in the unimpeded database on Zenodo.

Not every model-dataset combination exists on Zenodo. The `DatabaseExplorer` provides several ways to discover what's available.

In [ ]:
# Initialize the database explorer
dbe = DatabaseExplorer()

# Get lists of available models and datasets
print("Available models:")
print(dbe.models)
print(f"\nTotal: {len(dbe.models)} models")

print("\n" + "="*50)
print("\nAvailable datasets:")
print(dbe.datasets)
print(f"\nTotal: {len(dbe.datasets)} datasets")

### Discovering valid model-dataset combinations

Not all model-dataset pairs exist. Use `combinations`, `datasets_for()`, `models_for()`, and `is_available()` to find what's on Zenodo.

In [ ]:
# See all valid (model, dataset) pairs
print(f"Total valid combinations: {len(dbe.combinations)}\n")

# What datasets are available for a specific model?
print("Datasets available for 'lcdm':")
print(dbe.datasets_for('lcdm'))

# What models are available for a specific dataset?
print("\nModels available for 'planck_2018_plik':")
print(dbe.models_for('planck_2018_plik'))

# Check if a specific combination exists before downloading
print(f"\nis_available('lcdm', 'planck_2018_plik'): {dbe.is_available('lcdm', 'planck_2018_plik')}")
print(f"is_available('lcdm', 'fake_dataset'):     {dbe.is_available('lcdm', 'fake_dataset')}")

## 2. Downloading Samples

We can download nested sampling or MCMC chains for any model and dataset combination.

In [ ]:
# Choose parameters from the available lists
method = 'ns'  # 'ns' for nested sampling, 'mcmc' for MCMC
model = 'lcdm'
dataset = 'planck_2018_CamSpec'

# Download samples
samples = dbe.download_samples(method, model, dataset)

print(f"Downloaded samples shape: {samples.shape}")
print(f"\nFirst few parameters: {list(samples.columns)[:10]}")

In [ ]:
# Download run information (Cobaya and PolyChord settings)
info = dbe.download_info(method, model, dataset)
print("Run information keys:")
print(list(info.keys()))

### Visualizing Posterior Distributions

The samples are `anesthetic.NestedSamples` objects with built-in plotting capabilities.

In [ ]:
# Plot 1D marginalized posteriors for key cosmological parameters
# Common parameters: 'H0', 'omegabh2', 'omegach2', 'tau', 'ns', 'logA'
params_to_plot = ['H0', 'omegabh2', 'omegach2', 'tau']  # Adjust based on what's available

fig, axes = samples.plot_1d(params_to_plot)
plt.tight_layout()
plt.show()

In [ ]:
# Plot 2D contours (corner plot)
fig, axes = samples.plot_2d(['H0', 'omegabh2', 'omegach2'])
plt.tight_layout()
plt.show()

## 3. Computing Tension Statistics

The key feature of `unimpeded` is computing **tension statistics** between datasets. This quantifies how consistent different cosmological observations are with each other.

### Understanding Tension Metrics

The tension statistics include:

- **logR**: R statistic for dataset consistency (log-evidence based)
  - $\log R = \log Z_{AB} - \log Z_{A} - \log Z_{B}$
  
- **I**: Mutual information estimate (Kullback-Leibler divergence)
  - $\mathcal{I} = D_{KL}^{A} + D_{KL}^{B} - D_{KL}^{AB}$
  
- **logS**: Suspiciousness (log-likelihood based)
  - $\log S = \log L_{AB} - \log L_{A} - \log L_{B}$
  
- **d_G**: Gaussian model dimensionality of shared parameters
  - $d = d_{A} + d_{B} - d_{AB}$
  
- **p**: p-value for tension (higher = more consistent)
  
- **sigma**: Tension in units of σ
  - < 2σ: Datasets are consistent
  - 2-3σ: Mild tension
  - > 3σ: Significant tension

### Method 1: Simple Tension Calculation (Recommended)

The `tension_calculator` function provides a high-level interface that handles everything automatically.

In [ ]:
# Example 1: Tension between TWO datasets
tension_samples_2 = tension_calculator('ns',
                                       'lcdm',
                                       'planck_2018_CamSpec',
                                       'des_y1.joint',
                                       nsamples=1000)
print(tension_samples_2)
print(f"\nColumns: {tension_samples_2.columns.tolist()}")

In [ ]:
# Extract and interpret key results
if len(tension_samples_2) > 0:
    print("\n" + "="*50)
    print("TENSION RESULTS")
    print("="*50)
    print(f"Tension (σ): {tension_samples_2['sigma'].mean():.2f} ± {tension_samples_2['sigma'].std():.2f}")
    print(f"p-value: {tension_samples_2['p'].mean():.4f}")
    print(f"log R: {tension_samples_2['logR'].mean():.2f}")
    print(f"log S: {tension_samples_2['logS'].mean():.2f}")
    
    sigma = tension_samples_2['sigma'].mean()
    if sigma < 2:
        print("\nInterpretation: ✓ Datasets are consistent")
    elif sigma < 3:
        print("\nInterpretation: ⚠ Mild tension between datasets")
    else:
        print("\nInterpretation: ✗ Significant tension between datasets")

In [ ]:
# Plot tension distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot sigma distribution
axes[0].hist(tension_samples_2['sigma'], bins=50, alpha=0.7, color='steelblue', edgecolor='black')
axes[0].axvline(2, color='orange', linestyle='--', linewidth=2, label='2σ threshold')
axes[0].axvline(3, color='red', linestyle='--', linewidth=2, label='3σ threshold')
axes[0].set_xlabel('Tension (σ)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].set_title('Distribution of Tension', fontsize=14)
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot p-value distribution
axes[1].hist(tension_samples_2['p'], bins=50, alpha=0.7, color='coral', edgecolor='black')
axes[1].set_xlabel('p-value', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].set_title('Distribution of p-values', fontsize=14)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Example 2: Tension Between THREE Datasets

You can compute tension between multiple datasets simultaneously.

In [ ]:
# Tension between THREE datasets
tension_samples_3 = tension_calculator('ns',
                                       'lcdm',
                                       'planck_2018_CamSpec',
                                       'sn.pantheonplus',
                                       'bao.desi_dr2',
                                       nsamples=1000)
print(tension_samples_3)
print(f"\nTension (σ): {tension_samples_3['sigma'].mean():.2f} ± {tension_samples_3['sigma'].std():.2f}")

### Method 2: Two-Step Process with More Control

For more flexibility, you can separate the data download from the tension calculation.

In [ ]:
# Step 1: Download the required chains and prior information
print("STEP 1: Downloading chains and prior information...")
tension_data = download_tension_inputs('ns', 'lcdm', 'planck_2018_CamSpec', 'des_y1.joint')

print("\nDownloaded data structure:")
print(f"  - Joint samples: {type(tension_data['joint'])}")
print(f"  - Separate samples: {len(tension_data['separate'])} datasets")
print(f"  - Joint F factor: {tension_data['joint_f']:.4f}")
print(f"  - Separate F factors: {[f'{f:.4f}' for f in tension_data['separate_fs']]}")

In [ ]:
# Step 2: Calculate tension statistics using the downloaded data
print("STEP 2: Calculating tension statistics...")
tension_result = tension_stats(
    tension_data['joint'],
    *tension_data['separate'],
    joint_f=tension_data['joint_f'],
    separate_fs=tension_data['separate_fs'],
    nsamples=1000
)

print("\nTension Statistics Results:")
print(tension_result)
print(f"\nColumns: {tension_result.columns.tolist()}")

### Benefit of Caching

The `download_tension_inputs` function uses caching, so you can reuse the same data with different parameters without re-downloading.

In [ ]:
# This will use cached data (instant)
tension_data = download_tension_inputs('ns', 'lcdm', 'planck_2018_CamSpec', 'des_y1.joint')

# Calculate with different nsamples values
tension_result_500 = tension_stats(
    tension_data['joint'],
    *tension_data['separate'],
    joint_f=tension_data['joint_f'],
    separate_fs=tension_data['separate_fs'],
    nsamples=500
)

tension_result_1000 = tension_stats(
    tension_data['joint'],
    *tension_data['separate'],
    joint_f=tension_data['joint_f'],
    separate_fs=tension_data['separate_fs'],
    nsamples=1000
)

print("Results with nsamples=500:")
print(f"  σ = {tension_result_500['sigma'].mean():.2f} ± {tension_result_500['sigma'].std():.2f}")

print("\nResults with nsamples=1000:")
print(f"  σ = {tension_result_1000['sigma'].mean():.2f} ± {tension_result_1000['sigma'].std():.2f}")

### Method 3: Manual Method with Full Control

For complete control, you can download each component separately.

In [ ]:
# Define parameters
method = 'ns'
model = 'lcdm'
datasetA = 'planck_2018_CamSpec'
datasetB = 'des_y1.joint'
datasetAB = 'des_y1.joint+planck_2018_CamSpec'  # Joint dataset (alphabetically sorted)

print("Downloading samples for each dataset separately...")
samples_A = dbe.download_samples(method, model, datasetA)
print(f"  ✓ Downloaded samples for {datasetA}")

samples_B = dbe.download_samples(method, model, datasetB)
print(f"  ✓ Downloaded samples for {datasetB}")

samples_AB = dbe.download_samples(method, model, datasetAB)
print(f"  ✓ Downloaded samples for {datasetAB}")

In [ ]:
print("Downloading prior information for each dataset...")
prior_info_A = dbe.download_prior_info(model, datasetA)
prior_info_B = dbe.download_prior_info(model, datasetB)
prior_info_AB = dbe.download_prior_info(model, datasetAB)
print("  ✓ Prior info downloaded for all datasets")

# Calculate F correction factors
FA = prior_info_A['nprior'] / prior_info_A['ndiscarded']
FB = prior_info_B['nprior'] / prior_info_B['ndiscarded']
F_AB = prior_info_AB['nprior'] / prior_info_AB['ndiscarded']

print(f"\nCorrection factors:")
print(f"  F_A = {FA:.4f}")
print(f"  F_B = {FB:.4f}")
print(f"  F_AB = {F_AB:.4f}")

In [ ]:
# Calculate tension statistics
print("Calculating tension statistics...")
tension_result_manual = tension_stats(
    samples_AB,              # joint samples
    samples_A, samples_B,    # separate samples
    joint_f=F_AB,            # joint correction factor
    separate_fs=[FA, FB],    # separate correction factors
    nsamples=1000
)

print("\nTension Statistics Results (Manual Method):")
print(tension_result_manual)
print(f"\nColumns: {tension_result_manual.columns.tolist()}")

## 4. Comparing Across Models

Compare how tension varies across different cosmological models.

In [ ]:
# Compare tension across multiple models
models_to_compare = ['lcdm', 'klcdm', 'wlcdm']
dataset1 = 'planck_2018_CamSpec'
dataset2 = 'des_y1.joint'

tensions = {}
for model in models_to_compare:
    try:
        print(f"Computing tension for {model}...")
        result = tension_calculator('ns', model, dataset1, dataset2, nsamples=1000)
        if len(result) > 0:
            tensions[model] = {
                'mean': result['sigma'].mean(),
                'std': result['sigma'].std()
            }
    except Exception as e:
        print(f"  Could not compute tension for {model}: {e}")

print("\nTension Results:")
for model, values in tensions.items():
    print(f"  {model}: {values['mean']:.2f} ± {values['std']:.2f} σ")

In [ ]:
# Plot comparison
if tensions:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    models = list(tensions.keys())
    means = [tensions[m]['mean'] for m in models]
    stds = [tensions[m]['std'] for m in models]
    
    bars = ax.bar(models, means, yerr=stds, alpha=0.7, color='steelblue', 
                   capsize=5, edgecolor='black', linewidth=1.5)
    ax.axhline(y=2, color='orange', linestyle='--', linewidth=2, label='2σ threshold')
    ax.axhline(y=3, color='red', linestyle='--', linewidth=2, label='3σ threshold')
    
    ax.set_xlabel('Model', fontsize=14)
    ax.set_ylabel('Tension (σ)', fontsize=14)
    ax.set_title(f'Tension between {dataset1} and {dataset2}', fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Understanding Prior Information

The F correction factors account for prior volume effects in nested sampling.

In [ ]:
# Download and examine prior information
prior_info = dbe.download_prior_info('lcdm', 'planck_2018_CamSpec')

print("Prior Information:")
for key, value in prior_info.items():
    print(f"  {key}: {value}")

# Calculate F factor
F = prior_info['nprior'] / prior_info['ndiscarded']
print(f"\nCorrection factor F = nprior/ndiscarded = {F:.4f}")

## Summary

This notebook demonstrated three approaches to computing tension statistics:

1. **`tension_calculator`** - Simplest approach, handles everything automatically
2. **`download_tension_inputs` + `tension_stats`** - More control, with caching benefits
3. **Manual with `DatabaseExplorer`** - Full control over each step

Key takeaways:
- Tension quantifies consistency between different cosmological observations
- Results include σ values where > 3σ indicates significant tension
- F correction factors account for prior volume effects
- Caching makes repeated analyses very efficient

### Resources

- Documentation: http://unimpeded.readthedocs.io/
- GitHub: https://github.com/handley-lab/unimpeded
- Paper: Handley (2023), "unimpeded: cosmological inference across models and datasets"